In [0]:
# Install & Imports
%pip install yfinance --quiet
dbutils.library.restartPython()

In [0]:
# Imports & Config
import yfinance as yf
import pandas as pd
from datetime import datetime, date, timedelta
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *

CATALOG = "stock_analytics"
SCHEMA  = "gold"

TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "BRK-B", "JPM", "V"
]

print("✅ Imports done")

In [0]:
# Find latest date already in gold table
print("🔍 Checking latest dates in gold.daily_prices...\n")

latest_dates = (
    spark.table("stock_analytics.gold.daily_prices")
        .groupBy("ticker")
        .agg(F.max("trade_date").alias("latest_date"))
        .orderBy("ticker")
)

display(latest_dates)

# Get overall latest date across all tickers
overall_latest = (
    latest_dates
        .agg(F.min("latest_date").alias("fetch_from"))  # min = most conservative
        .collect()[0]["fetch_from"]
)

# Start from day AFTER latest date
START_DATE = (overall_latest + timedelta(days=1)).strftime("%Y-%m-%d")
END_DATE   = date.today().strftime("%Y-%m-%d")

print(f"\n📅 Latest date in table : {overall_latest}")
print(f"📅 Fetching from        : {START_DATE}")
print(f"📅 Fetching to          : {END_DATE}")

# Guard — if already up to date, stop here
if START_DATE > END_DATE:
    print("\n✅ Table already up to date — nothing to fetch!")
    dbutils.notebook.exit("UP_TO_DATE")

In [0]:
# Fetch incremental data from yfinance
print(f"🚀 Fetching new data: {START_DATE} → {END_DATE}\n")

new_data = []

for ticker in TICKERS:
    print(f"  📡 Fetching: {ticker}")
    try:
        stock = yf.Ticker(ticker)
        df    = stock.history(start=START_DATE, end=END_DATE)

        if df.empty:
            print(f"    ⚠️  No new data for {ticker}")
            continue

        df = df.reset_index()
        df["ticker"]      = ticker
        df["trade_date"]  = df["Date"].dt.date
        df["open"]        = df["Open"].round(6)
        df["high"]        = df["High"].round(6)
        df["low"]         = df["Low"].round(6)
        df["close"]       = df["Close"].round(6)
        df["volume"]      = df["Volume"].astype(float)
        df["data_source"] = "yfinance_incremental"

        df = df[[
            "ticker", "trade_date", "open", "high",
            "low", "close", "volume", "data_source"
        ]]

        new_data.append(df)
        print(f"    ✅ {len(df)} new rows")

    except Exception as e:
        print(f"    ❌ Failed {ticker}: {str(e)}")

if len(new_data) == 0:
    print("\n✅ No new data found — all tickers up to date!")
    dbutils.notebook.exit("NO_NEW_DATA")

new_df = pd.concat(new_data, ignore_index=True)
print(f"\n📊 Total new rows fetched: {len(new_df)}")

In [0]:
# Add derived columns to new data

schema = StructType([
    StructField("ticker",      StringType(), True),
    StructField("trade_date",  DateType(),   True),
    StructField("open",        DoubleType(), True),
    StructField("high",        DoubleType(), True),
    StructField("low",         DoubleType(), True),
    StructField("close",       DoubleType(), True),
    StructField("volume",      DoubleType(), True),
    StructField("data_source", StringType(), True)
])

new_spark_df = spark.createDataFrame(new_df, schema=schema)

# ── Window for moving averages ───────────────────
# Need full history for accurate moving averages
# So union new data with existing table first
existing_df = spark.table("stock_analytics.gold.daily_prices").select(
    "ticker", "trade_date", "open", "high",
    "low", "close", "volume", "data_source"
)

full_df = existing_df.unionByName(new_spark_df)

window_7d  = Window.partitionBy("ticker").orderBy("trade_date").rowsBetween(-6,  0)
window_30d = Window.partitionBy("ticker").orderBy("trade_date").rowsBetween(-29, 0)

enriched_full = (
    full_df
        .withColumn("daily_return_pct",
            F.round((F.col("close") - F.col("open")) / F.col("open") * 100, 4))
        .withColumn("price_range",
            F.round(F.col("high") - F.col("low"), 4))
        .withColumn("moving_avg_7d",
            F.round(F.avg("close").over(window_7d), 4))
        .withColumn("moving_avg_30d",
            F.round(F.avg("close").over(window_30d), 4))
        .withColumn("ingested_at", F.current_timestamp())
)

# ── Keep only NEW rows for merging ───────────────
new_enriched = enriched_full.filter(
    F.col("trade_date") > F.lit(overall_latest)
)

print(f"✅ Enriched new rows: {new_enriched.count()}")
display(new_enriched.orderBy("ticker", "trade_date"))

In [0]:
# MERGE new data into gold.daily_prices

# Register new data as temp view for MERGE
new_enriched.createOrReplaceTempView("new_daily_prices")

spark.sql("""
    MERGE INTO stock_analytics.gold.daily_prices AS target
    USING new_daily_prices AS source
    ON target.ticker     = source.ticker
    AND target.trade_date = source.trade_date

    -- If row exists → update it (handles corrections)
    WHEN MATCHED THEN UPDATE SET
        target.open             = source.open,
        target.high             = source.high,
        target.low              = source.low,
        target.close            = source.close,
        target.volume           = source.volume,
        target.daily_return_pct = source.daily_return_pct,
        target.price_range      = source.price_range,
        target.moving_avg_7d    = source.moving_avg_7d,
        target.moving_avg_30d   = source.moving_avg_30d,
        target.data_source      = source.data_source,
        target.ingested_at      = source.ingested_at

    -- If row doesn't exist → insert it
    WHEN NOT MATCHED THEN INSERT *
""")

new_total = spark.table("stock_analytics.gold.daily_prices").count()
print(f"✅ MERGE complete — gold.daily_prices")
print(f"   Previous rows : {new_total - new_enriched.count()}")
print(f"   New rows added: {new_enriched.count()}")
print(f"   Total rows now: {new_total}")

In [0]:
# Recompute stock_volatility from full updated table

full_updated = spark.table("stock_analytics.gold.daily_prices")

volatility_df = (
    full_updated
        .groupBy("ticker")
        .agg(
            F.round(F.stddev("daily_return_pct"), 4) .alias("volatility_stddev"),
            F.round(F.avg("daily_return_pct"),    4) .alias("avg_daily_return"),
            F.round(F.max("daily_return_pct"),    4) .alias("best_day_return"),
            F.round(F.min("daily_return_pct"),    4) .alias("worst_day_return"),
            F.count("*")                             .alias("total_trading_days"),
            F.min("trade_date")                      .alias("data_from"),
            F.max("trade_date")                      .alias("data_to")
        )
        .withColumn("risk_tier",
            F.when(F.col("volatility_stddev") > 3,   "🔴 High")
             .when(F.col("volatility_stddev") > 1.5,  "🟡 Medium")
             .otherwise(                              "🟢 Low")
        )
)

(
    volatility_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("stock_analytics.gold.stock_volatility")
)

print("✅ stock_volatility recomputed & updated")
display(volatility_df.orderBy("volatility_stddev", ascending=False))

In [0]:
# Recompute monthly_performance from full updated table

monthly_agg = (
    full_updated
        .withColumn("month", F.date_format("trade_date", "yyyy-MM"))
        .groupBy("ticker", "month")
        .agg(
            F.round(F.sum("daily_return_pct"),  2).alias("monthly_return_pct"),
            F.round(F.max("daily_return_pct"),  2).alias("best_single_day"),
            F.round(F.min("daily_return_pct"),  2).alias("worst_single_day"),
            F.round(F.avg("close"),             4).alias("avg_close"),
            F.count("*")                          .alias("trading_days")
        )
)

window_rank = Window.partitionBy("month").orderBy(F.desc("monthly_return_pct"))
monthly_df  = monthly_agg.withColumn("rank", F.rank().over(window_rank))

(
    monthly_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("stock_analytics.gold.monthly_performance")
)

print("✅ monthly_performance recomputed & updated")
print(f"   Total rows: {monthly_df.count()}")

In [0]:
# Final run summary
print("=" * 55)
print("✅ INCREMENTAL LOAD COMPLETE")
print("=" * 55)

for table in ["daily_prices", "stock_volatility", "monthly_performance"]:
    full_path = f"stock_analytics.gold.{table}"
    count     = spark.table(full_path).count()
    latest    = spark.table(full_path).agg(
                    F.max("trade_date" if table != "stock_volatility" 
                          else "data_to")).collect()[0][0]
    print(f"\n📊 {table}")
    print(f"   Rows        : {count}")
    print(f"   Latest date : {latest}")

print(f"\n📅 Run completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("🚀 All Gold tables are up to date!")